# 02 CoM and Bounding Box

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`Automatic_Centering_Transformer.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/Automatic_Centering_Transformer.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`Automatic_Centering_Experiment.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/Automatic_Centering_Experiment.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`CoM_Decision_Diagnostic.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/CoM_Decision_Diagnostic.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`CoM_Translation_Augmentation_Experiment.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/CoM_Translation_Augmentation_Experiment.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`exp006_size_com_comparison.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp006_size_com_comparison.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Normal/scripts/Automatic_Centering_Transformer.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'eb780003845c477747d411265b5bacae0d9a1481'}, '05_model_optimize/MLP/Normal/scripts/Automatic_Centering_Experiment.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'b4d8935e36a10acbbec5a81e62289bef7358a24f'}, '05_model_optimize/MLP/Normal/scripts/CoM_Decision_Diagnostic.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '5145746b6f78fb02dfd87a9f793bedaa3de49f38'}, '05_model_optimize/MLP/Normal/scripts/CoM_Translation_Augmentation_Experiment.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '68bd0266b898cba4bbb0f400d04d5b5ce34daf8d'}, '05_model_optimize/MLP/Normal/scripts/exp006_size_com_comparison.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '808dcf7e34164dd86890f846521bd481acc2eb70'}}

## Automatic_Centering_Transformer

원본: [05_model_optimize/MLP/Normal/scripts/Automatic_Centering_Transformer.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/Automatic_Centering_Transformer.py)  
Git blob: `eb780003845c477747d411265b5bacae0d9a1481`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

EXP유004: stateless image-by-image geometric centering, label-free.

Center-of-mass and bounding-box center share:
- an identical foreground threshold (32 on raw grayscale [0,255]);
- a fixed geometric image target (13.5,13.5);
- integer-pixel translation (no interpolation), zero-fill and NO wrapping;
- no fitted parameters, class labels, writer IDs, or Handmade statistics.

This module must remain importable by its stable name for joblib Pipeline loading.



In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/Automatic_Centering_Transformer.py
#!/usr/bin/env python3
"""EXP유004: stateless image-by-image geometric centering, label-free.

Center-of-mass and bounding-box center share:
- an identical foreground threshold (32 on raw grayscale [0,255]);
- a fixed geometric image target (13.5,13.5);
- integer-pixel translation (no interpolation), zero-fill and NO wrapping;
- no fitted parameters, class labels, writer IDs, or Handmade statistics.

This module must remain importable by its stable name for joblib Pipeline loading.
"""
from __future__ import annotations

import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

THRESHOLD = 32.0
TARGET = (13.5, 13.5)  # (x,y), 28x28 geometric center


def raw_centers(X: np.ndarray, method: str, threshold: float = THRESHOLD):
    """Return foreground-dependent (x,y) coordinates for every source image."""
    if method not in ("mass", "bbox"):
        raise ValueError(f"Unknown method {method!r}")
    a = np.asarray(X, dtype=np.float32).reshape(-1, 28, 28)
    if not np.isfinite(a).all():
        raise ValueError("Nonfinite source image intensity")
    mask = a >= threshold
    valid = mask.any(axis=(1, 2))
    if not valid.all():
        raise ValueError(f"{int((~valid).sum())} empty foreground masks at threshold {threshold}")

    if method == "mass":
        weight = np.where(mask, a, 0).astype(np.float64, copy=False)
        total = weight.sum(axis=(1, 2))
        cx = (weight * np.arange(28, dtype=float)[None, None, :]).sum(axis=(1, 2))/total
        cy = (weight * np.arange(28, dtype=float)[None, :, None]).sum(axis=(1, 2))/total
    else:
        xpresent = mask.any(axis=1)
        ypresent = mask.any(axis=2)
        xmin = xpresent.argmax(axis=1)
        xmax = 27 - xpresent[:, ::-1].argmax(axis=1)
        ymin = ypresent.argmax(axis=1)
        ymax = 27 - ypresent[:, ::-1].argmax(axis=1)
        cx = (xmin+xmax)/2
        cy = (ymin+ymax)/2
    return cx, cy


def center_shifts(X: np.ndarray, method: str):
    cx, cy = raw_centers(X, method)
    dx = np.rint(TARGET[0] - cx).astype(np.int32)
    dy = np.rint(TARGET[1] - cy).astype(np.int32)
    return np.column_stack([dx, dy])


def center_batch(X: np.ndarray, method: str):
    a = np.asarray(X, dtype=np.float32).reshape(-1,28,28)
    shifts = center_shifts(a, method)
    result = np.zeros_like(a)
    for dx,dy in np.unique(shifts,axis=0):
        dx,dy = int(dx),int(dy)
        ids = np.flatnonzero((shifts[:,0]==dx)&(shifts[:,1]==dy))
        ys0,ys1=max(0,-dy), min(28,28-dy)
        xs0,xs1=max(0,-dx), min(28,28-dx)
        if ys0>=ys1 or xs0>=xs1:
            continue
        result[ids,ys0+dy:ys1+dy,xs0+dx:xs1+dx] = a[ids,ys0:ys1,xs0:xs1]
    return result.reshape(-1,784)


class AutomaticCentering(BaseEstimator, TransformerMixin):
    def __init__(self, method: str = "mass"):
        self.method = method

    def fit(self, X, y=None):
        if self.method not in ("mass","bbox"):
            raise ValueError(self.method)
        return self

    def transform(self, X):
        return center_batch(X, self.method)


def test_centering():
    """Geometry, sign, data-integrity, no-wrap, input immutability checks."""
    a=np.zeros((3,28,28),dtype=np.float32)
    a[0,6:10,4:8]=255
    a[1,18:22,20:24]=255
    a[2,12:16,10:14]=255
    raw=a.copy()
    for method in ("mass","bbox"):
        out=center_batch(a,method).reshape(-1,28,28)
        before=raw_centers(a,method)
        after=raw_centers(out,method)
        for i in range(3):
            assert abs(after[0][i]-13.5)<=0.50001
            assert abs(after[1][i]-13.5)<=0.50001
            assert abs(float(out[i].sum())-float(a[i].sum()))<1e-5
        assert np.allclose(raw,a), "Input was mutated"
        assert not np.any(out < 0)
        assert np.unique(out).tolist()==[0.0,255.0]
    edge=np.zeros((1,28,28),dtype=np.float32)
    edge[0,13,0]=255
    for method in ("mass","bbox"):
        centered=center_batch(edge,method).reshape(28,28)
        assert centered[13,14]==255
        assert centered[13,0]==0
    assert np.array_equal(center_batch(a,"mass"),center_batch(a,"mass"))
    print("CENTERING SELF-TEST PASSED: label-free methods and zero-filled integer shifts",flush=True)


## Automatic_Centering_Experiment

원본: [05_model_optimize/MLP/Normal/scripts/Automatic_Centering_Experiment.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/Automatic_Centering_Experiment.py)  
Git blob: `b4d8935e36a10acbbec5a81e62289bef7358a24f`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

EXP유004 review-only: Normal vs centered MLPs, paired Official Val + Handmade.

Never reads official Test. Does not publish results or commit trained models.
Uses EXACT preexisting baseline (EXP유002 frozen joblib) and refits two
unchanged sklearn MLPClassifier(random_state=42) on the same 76,800 official
Training examples, adding only a stateless centering transform to each new
Pipeline. Handmade is NEVER used in fit or preprocessing parameter selection.

CLI 인자 (정적 추출): `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/Automatic_Centering_Experiment.py
#!/usr/bin/env python3
"""EXP유004 review-only: Normal vs centered MLPs, paired Official Val + Handmade.

Never reads official Test. Does not publish results or commit trained models.
Uses EXACT preexisting baseline (EXP유002 frozen joblib) and refits two
unchanged sklearn MLPClassifier(random_state=42) on the same 76,800 official
Training examples, adding only a stateless centering transform to each new
Pipeline. Handmade is NEVER used in fit or preprocessing parameter selection.
"""
from __future__ import annotations

import argparse
import csv
import gzip
import hashlib
import json
import platform
import sys
import time
import warnings
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import sklearn
import scipy
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline

HERE=Path(__file__).resolve().parent
ROOT=HERE.parents[3]
sys.path.insert(0,str(HERE))
from Pixcel_Attribution import (fixed_split, DEFAULT_DATA, read_handmade,
    prior_official_cm, prior_handmade, LABELS, build_baseline)
from Automatic_Centering_Transformer import (AutomaticCentering,
    raw_centers, center_shifts, center_batch, test_centering, TARGET, THRESHOLD)
from Input_Sensitivity import transform, Variant
# Pixcel_Attribution imports 'common' which exposes the actual baseline scaler.
from common import pipeline
from pixel_transform import PixelScaler255

STAGE3_MODEL=ROOT/"05_model_optimize"/"MLP"/"Normal"/"results"/"Input_Sensitivity"/"normal_mlp_refit_stage3.joblib"
EXPECTED_STAGE3_SHA="9b54ba9c60e389575200d88ad0b93c47ec47f9c673eebeb755ff47e3ebe23b0a"
CONDITIONS=("Normal","Center_of_Mass","Bounding_Box_Center")
CONFUSIONS=(("5","F"),("B","8"),("0","D"),("9","4"),("E","C"),("4","A"))
WRITERS=(41,42,43,44)

def checksum(p):
    with p.open("rb") as f:
        return hashlib.file_digest(f,"sha256").hexdigest()

def write_csv(p,rows):
    if not rows:
        raise ValueError("No rows to write "+str(p))
    with p.open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)

def score_row(domain,method,y,pred,mask,group_type,group,base_pred):
    y,pred,base_pred=y[mask],pred[mask],base_pred[mask]
    a=(pred==y)
    old=(base_pred==y)
    n=len(y)
    assert n>0
    cm=confusion_matrix(y,pred,labels=np.arange(16))
    return {
        "dataset":domain, "method":method,"group_type":group_type, "group":str(group),
        "n":n,"accuracy":float(a.mean()),
        "macro_f1":float(f1_score(y,pred,average="macro",labels=np.arange(16),zero_division=0)),
        "macro_precision":float(precision_score(y,pred,average="macro",labels=np.arange(16),zero_division=0)),
        "macro_recall":float(recall_score(y,pred,average="macro",labels=np.arange(16),zero_division=0)),
        "correct":int(a.sum()),
        "delta_correct_vs_baseline":int(a.sum()-old.sum()),
        "accuracy_delta_pp_vs_baseline":float(100*(a.sum()-old.sum())/n),
        "corrected_errors":int((~old & a).sum()),
        "induced_errors":int((old & ~a).sum()),
        "changed_predictions":int((base_pred!=pred).sum()),
        "prediction_change_rate":float((base_pred!=pred).mean()),
        "original_baseline_correct":int(old.sum()),
        "confusion_off_diagonal":int(n-np.trace(cm))
    }

def per_group(domain, y, writers, predictions):
    baseline=predictions["Normal"]
    ones=np.ones(len(y),dtype=bool)
    rows=[]
    for name,pred in predictions.items():
        rows.append(score_row(domain,name,y,pred,ones,"all","all",baseline))
        for c in range(16):
            rows.append(score_row(domain,name,y,pred,y==c,"class",LABELS[c],baseline))
        if domain=="Handmade":
            for kind,mask in [
                ("digits",y<10),("letters",y>=10)
            ]:
                rows.append(score_row(domain,name,y,pred,mask,"category",kind,baseline))
            for writer in WRITERS:
                m=writers==writer
                rows.append(score_row(domain,name,y,pred,m,"writer",str(writer),baseline))
                for typ,cm in [("digits",y<10),("letters",y>=10)]:
                    rows.append(score_row(domain,name,y,pred,m&cm,"writer_category",f"{writer}_{typ}",baseline))
    return rows

def alignment_rows(X, domain):
    """Quantify centering effects without labels or writer information."""
    a=np.asarray(X,dtype=np.float32).reshape(-1,784)
    rows=[]
    pixel_mass=a.sum(axis=1,dtype=np.float64)
    for method in ("mass","bbox"):
        shifts=center_shifts(a,method)
        aligned=center_batch(a,method)
        old_x,old_y=raw_centers(a,method)
        new_x,new_y=raw_centers(aligned,method)
        old_err=np.sqrt((old_x-TARGET[0])**2+(old_y-TARGET[1])**2)
        new_err=np.sqrt((new_x-TARGET[0])**2+(new_y-TARGET[1])**2)
        lost=np.maximum(0,pixel_mass-aligned.sum(axis=1,dtype=np.float64))
        fraction=lost/np.maximum(pixel_mass,1)
        rows.append({
            "dataset":domain,"centering_method":method,"n":len(a),
            "mean_center_error_before_px":float(old_err.mean()),
            "mean_center_error_after_px":float(new_err.mean()),
            "median_center_error_after_px":float(np.median(new_err)),
            "p95_center_error_after_px":float(np.quantile(new_err,.95)),
            "mean_absolute_dx":float(np.abs(shifts[:,0]).mean()),
            "mean_absolute_dy":float(np.abs(shifts[:,1]).mean()),
            "max_absolute_shift":int(np.abs(shifts).max()),
            "images_with_nonzero_ink_loss":int(np.count_nonzero(lost>0.01)),
            "fraction_with_nonzero_ink_loss":float(np.mean(lost>0.01)),
            "mean_lost_ink_fraction":float(fraction.mean()),
            "p95_lost_ink_fraction":float(np.quantile(fraction,.95)),
            "max_lost_ink_fraction":float(fraction.max()),
            "images_with_remaining_center_error_over_1px":int((new_err>1).sum()),
        })
    return rows

def paired_predictions(domain,y,writers,indices,preds,out_file):
    names=list(preds)
    assert names==list(CONDITIONS)
    with gzip.open(out_file,"wt",encoding="utf-8",newline="") as f:
        cols=["dataset","writer","image_index","true"]
        for name in names:
            cols.extend([f"{name}_pred",f"{name}_correct"])
        w=csv.DictWriter(f,fieldnames=cols)
        w.writeheader()
        for k in range(len(y)):
            r={"dataset":domain,"writer":int(writers[k]),
               "image_index":int(indices[k]),"true":LABELS[int(y[k])]}
            for name in names:
                r[f"{name}_pred"]=LABELS[int(preds[name][k])]
                r[f"{name}_correct"]=int(preds[name][k]==y[k])
            w.writerow(r)

def paired_confusions(y,preds,domain):
    rows=[]
    for name,values in preds.items():
        cm=confusion_matrix(y,values,labels=np.arange(16))
        for i,j in np.ndindex(cm.shape):
            rows.append({"dataset":domain,"method":name,
                         "true":LABELS[i],"predicted":LABELS[j],"count":int(cm[i,j])})
    return rows

def plot_scores(rows,out):
    allrows=[r for r in rows if r["group_type"]=="all"]
    fig, ax=plt.subplots(figsize=(9,4.7))
    x=np.arange(len(CONDITIONS))
    for j,domain in enumerate(("OfficialVal","Handmade")):
        vals=[next(r["accuracy"] for r in allrows if r["dataset"]==domain and r["method"]==name) for name in CONDITIONS]
        ax.bar(x+(-.19 if j==0 else .19), np.asarray(vals)*100, width=.37,label=domain)
        for xi,v in zip(x+(-.19 if j==0 else .19),vals):
            ax.text(xi,100*v+.35,f"{100*v:.2f}%",ha="center",fontsize=9)
    ax.set_xticks(x,["Normal","Center of Mass","Bounding Box"])
    ax.set_ylim(0,105)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("EXP004 — fixed Normal vs centering-before-training MLPs")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out/"accuracy_comparison.png",dpi=145)
    plt.close(fig)

def summary_text(rows,align,warnings_by_model,off_cm_delta):
    wanted=[r for r in rows if r["group_type"]=="all"]
    txt=["# EXP유004 — Automatic_Centering (review-only, measured results)","",
         "**No official Test, no professor Test, and no Handmade training or hyperparameter selection.**",
         "Same 76,800 Official Training examples and unchanged MLPClassifier(random_state=42).",
         "The current Normal model is the previously validated frozen Stage 3 checkpoint; the two centering models are freshly fit.",
         "Training and inference use the SAME centering method in each new Pipeline.",
         "Preset geometric target = (13.5,13.5) pixel coordinates; threshold=32; integer-pixel translation, no interpolation.",
         "",
         "## Official Val and Handmade — paired accuracy / corrections",
         "| Dataset | Model | Accuracy | Δ baseline (pp) | Corrected | Newly wrong | Macro F1 |",
         "|---|---|---:|---:|---:|---:|---:|"]
    for domain in ("OfficialVal","Handmade"):
        for name in CONDITIONS:
            r=next(a for a in wanted if a["dataset"]==domain and a["method"]==name)
            txt.append(f"| {domain} | {name} | {r['accuracy']:.4%} | {r['accuracy_delta_pp_vs_baseline']:+.3f} | {r['corrected_errors']} | {r['induced_errors']} | {r['macro_f1']:.4f} |")
    txt+=["","## Alignment distortion and consistency",
          "| Data | Center | Mean center error before→after | Images with ink loss | Mean lost ink fraction |",
          "|---|---|---|---:|---:|"]
    for a in align:
        txt.append(f"| {a['dataset']} | {a['centering_method']} | {a['mean_center_error_before_px']:.3f}→{a['mean_center_error_after_px']:.3f}px | {a['images_with_nonzero_ink_loss']}/{a['n']} | {a['mean_lost_ink_fraction']:.5f} |")
    txt+=["","## Generalization/interpretation safeguards",
          "- This four-writer Handmade set has already been repeatedly inspected in EXP유001-003. Improvements here are **exploratory**, NOT evidence of unseen-writer Test accuracy.",
          "- Official Val was also used in historical research; do not claim completely untouched test performance.",
          "- Since the baseline is already strong on official Validation, inspect the domain gap and class/writer strata, not a tiny official accuracy gain alone.",
          "- Centering can shift, clip or reshape which input pixels the MLP sees. Here only integer translation with zero-fill is used: measure ink clipping separately.",
          "- Class labels/writer IDs are for evaluation ONLY. Neither centering transformer uses them.",
          "- EXP유004 does not determine a final deployed preprocessing policy; await review and independent writer evaluation.",
          f"- Historical baseline Official CM L1 difference: {off_cm_delta}.",
          "",
          "## Files",
          "- exp004_group_metrics.csv: official and Handmade full/class/writer/category paired measures.",
          "- exp004_predictions.csv.gz: per-image predictions (OfficialVal + Handmade) for all 3 models.",
          "- exp004_confusions.csv: 16×16 confusion entries for all models and both domains.",
          "- exp004_alignment_stats.csv: target-centering accuracy, translation and stroke clipping.",
          "- exp004_typical_confusions.csv: selected 5/F, B/8, 0/D confusion changes.",
          "- accuracy_comparison.png: performance chart.",
          "- exp004_run_metadata.json: SHA256 hashes, software versions, training times and fit boundaries.",
          "- *_{mass,bbox}.joblib: trained candidate Pipelines, user review artifact only.",
          ""]
    return "\n".join(txt)

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--out",type=Path,required=True)
    args=ap.parse_args()
    started=time.monotonic()
    out=args.out.resolve()
    out.mkdir(parents=True,exist_ok=True)
    test_centering()
    assert checksum(STAGE3_MODEL)==EXPECTED_STAGE3_SHA
    xtr,ytr,xva,yva,sha=fixed_split(DEFAULT_DATA)
    xhm,yhm,whm,ih=read_handmade()
    ntr,nva,nh=len(ytr),len(yva),len(yhm)
    assert (ntr,nva,nh)==(76800,19200,2560)
    print("DATA VERIFIED: Official Training 76800; Official Validation 19200; Handmade 2560, 4 writers",flush=True)
    baseline=joblib.load(STAGE3_MODEL)  # Checksum verified before deserialization
    assert baseline.named_steps["model"].get_params()==build_baseline()[0].get_params()
    predv0=baseline.predict(xva)
    predh0=baseline.predict(xhm)
    cm_l1=int(np.abs(confusion_matrix(yva,predv0,labels=np.arange(16))-prior_official_cm()).sum())
    hist=prior_handmade()
    hmis=sum(hist[(int(w),int(idx))]!=(LABELS[int(y)],LABELS[int(p)])
             for w,idx,y,p in zip(whm,ih,yhm,predh0))
    assert cm_l1==0 and hmis==0
    assert abs(accuracy_score(yva,predv0)-.9731770833333333)<1e-12
    assert abs(accuracy_score(yhm,predh0)-.628515625)<1e-12
    print("BASELINE VERIFIED: Official Val 97.317708%, Handmade 62.8515625%, CM L1=0, handwritten mismatch=0",flush=True)

    predict={"OfficialVal":{"Normal":predv0}, "Handmade":{"Normal":predh0}}
    training_times={}
    run_warnings={}
    for name,method in (("Center_of_Mass","mass"),("Bounding_Box_Center","bbox")):
        print(f"TRAINING {name} same split + unchanged MLP hyperparameters...",flush=True)
        estimator,scaled=build_baseline()
        assert scaled
        estimator_params=estimator.get_params()
        model=Pipeline([
            ("centering",AutomaticCentering(method=method)),
            ("scale_0_1",PixelScaler255()),
            ("model",estimator),
        ])
        t0=time.monotonic()
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter("always")
            model.fit(xtr,ytr)
        elapsed=time.monotonic()-t0
        training_times[name]=float(elapsed)
        run_warnings[name]=[f"{w.category.__name__}: {w.message}" for w in captured]
        assert model.named_steps["model"].get_params()==estimator_params
        pv=model.predict(xva)
        ph=model.predict(xhm)
        predict["OfficialVal"][name]=pv
        predict["Handmade"][name]=ph
        joblib.dump(model,out/f"exp004_{method}.joblib",compress=3)
        print(f"TRAINED {name} in {elapsed:.1f}s: Official={accuracy_score(yva,pv):.6%}, Handmade={accuracy_score(yhm,ph):.6%}",flush=True)

    metrics=[]
    for domain,y,writers in (("OfficialVal",yva,np.full(nva,-1,dtype=np.int32)),
                              ("Handmade",yhm,whm)):
        metrics.extend(per_group(domain,y,writers,predict[domain]))
    write_csv(out/"exp004_group_metrics.csv",metrics)
    with gzip.open(out/"exp004_predictions.csv.gz","wt",encoding="utf-8",newline="") as f:
        cols=["dataset","writer","image_index","true"]
        for name in CONDITIONS:
            cols += [f"{name}_pred", f"{name}_correct"]
        csvwriter=csv.DictWriter(f,fieldnames=cols)
        csvwriter.writeheader()
        for domain,y,writers,indices in [
            ("OfficialVal",yva,np.full(nva,-1,dtype=np.int32),np.arange(nva)),
            ("Handmade",yhm,whm,ih)]:
            preds=predict[domain]
            for k in range(len(y)):
                row={"dataset":domain,"writer":int(writers[k]),"image_index":int(indices[k]),
                     "true":LABELS[int(y[k])]}
                for name in CONDITIONS:
                    row[f"{name}_pred"]=LABELS[int(preds[name][k])]
                    row[f"{name}_correct"]=int(preds[name][k]==y[k])
                csvwriter.writerow(row)
    confusions=[]
    for domain,y in (("OfficialVal",yva),("Handmade",yhm)):
        confusions.extend(paired_confusions(y,predict[domain],domain))
    write_csv(out/"exp004_confusions.csv",confusions)

    important=[]
    for domain,y in (("OfficialVal",yva),("Handmade",yhm)):
        for name in CONDITIONS:
            cm=confusion_matrix(y,predict[domain][name],labels=np.arange(16))
            for actual,predicted in CONFUSIONS:
                a,b=LABELS.index(actual),LABELS.index(predicted)
                important.append({"dataset":domain,"method":name,"actual":actual,
                                  "predicted":predicted,"errors":int(cm[a,b])})
    write_csv(out/"exp004_typical_confusions.csv",important)
    print("Calculating accuracy of centering and foreground ink-loss diagnostics...",flush=True)
    align=[]
    for domain,x in (("OfficialTrain",xtr),("OfficialVal",xva),("Handmade",xhm)):
        align.extend(alignment_rows(x,domain))
    write_csv(out/"exp004_alignment_stats.csv",align)
    plot_scores(metrics,out)
    meta={
        "study":"EXP유004 Automatic_Centering",
        "status":"review-only; NOT published to Normal results/logs",
        "baseline_model_sha256":EXPECTED_STAGE3_SHA,
        "official_npz_sha256":sha,
        "official_test_read":False,
        "professor_test_read":False,
        "handmade_used_in_fit":False,
        "handmade_used_for_preprocessing_policy":False,
        "official_train":ntr,"official_validation":nva,"handmade":nh,
        "handmade_writer_ids":list(WRITERS),
        "normal_previously_frozen":True,
        "new_models_fit_on_official_train":["Center_of_Mass","Bounding_Box_Center"],
        "baseline_historical_official_cm_l1":cm_l1,
        "baseline_historical_handmade_prediction_mismatches":hmis,
        "center_policy":{"methods":["mass","bbox"],"target_xy":list(TARGET),
                         "threshold":THRESHOLD,"translation":"integer np.rint shifts, zero-filled, no wrap or interpolation",
                         "handmade_tuning":False},
        "model_hyperparameters":{k:repr(v) for k,v in build_baseline()[0].get_params().items()},
        "candidate_training_seconds":training_times,
        "candidate_training_warnings":run_warnings,
        "software":{"python":platform.python_version(),"numpy":np.__version__,
                    "sklearn":sklearn.__version__,"scipy":scipy.__version__,
                    "matplotlib":matplotlib.__version__},
        "elapsed_seconds":float(time.monotonic()-started),
    }
    (out/"exp004_run_metadata.json").write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding="utf-8")
    (out/"SUMMARY.md").write_text(summary_text(metrics,align,run_warnings,cm_l1),encoding="utf-8")
    print("EXP004 COMPARISON COMPLETE",flush=True)
    print("------ MAIN RESULTS ------",flush=True)
    for domain in ("OfficialVal","Handmade"):
        for name in CONDITIONS:
            r=next(rr for rr in metrics if rr["dataset"]==domain and rr["method"]==name and rr["group_type"]=="all")
            print(f"{domain} {name}: acc={r['accuracy']:.8%}, delta_pp={r['accuracy_delta_pp_vs_baseline']:+.4f}, corrected={r['corrected_errors']}, induced={r['induced_errors']}, macro_F1={r['macro_f1']:.5f}",flush=True)
    for method in ("mass","bbox"):
        for a in align:
            if a["centering_method"]==method:
                print(f"ALIGNMENT {a['dataset']} {method}: center_err_px {a['mean_center_error_before_px']:.4f}->{a['mean_center_error_after_px']:.4f}, ink_loss={a['images_with_nonzero_ink_loss']}/{a['n']}",flush=True)
    print(f"TRAINING TIMES={training_times}; TOTAL SEC={meta['elapsed_seconds']:.1f}",flush=True)

if __name__=="__main__":
    main()


## CoM_Decision_Diagnostic

원본: [05_model_optimize/MLP/Normal/scripts/CoM_Decision_Diagnostic.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/CoM_Decision_Diagnostic.py)  
Git blob: `5145746b6f78fb02dfd87a9f793bedaa3de49f38`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

EXP유004 follow-up (review only): Why CoM boosts digits but hurts letters.

Two *controlled* contrasts:
  1) Same frozen baseline weights, RAW vs CoM-shifted pixels (movement effect).
  2) Same CoM-shifted pixels, frozen Normal vs CoM-trained weights (fit effect).
The CoM-trained model fed RAW pixels is a diagnostic off-training-distribution
control, not a deployable policy.

Fixed class-logit margin Integrated Gradients on selected *real* transitions:
true 5 vs F, true B vs 8. Always same black 

CLI 인자 (정적 추출): `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/CoM_Decision_Diagnostic.py
#!/usr/bin/env python3
"""EXP유004 follow-up (review only): Why CoM boosts digits but hurts letters.

Two *controlled* contrasts:
  1) Same frozen baseline weights, RAW vs CoM-shifted pixels (movement effect).
  2) Same CoM-shifted pixels, frozen Normal vs CoM-trained weights (fit effect).
The CoM-trained model fed RAW pixels is a diagnostic off-training-distribution
control, not a deployable policy.

Fixed class-logit margin Integrated Gradients on selected *real* transitions:
true 5 vs F, true B vs 8. Always same black reference and target sign.
No model fitting, no Test, no new hyperparameter choices, no publication.
"""
from __future__ import annotations

import argparse
import csv
import gzip
import hashlib
import json
import platform
import sys
from collections import Counter
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.metrics import accuracy_score, confusion_matrix

HERE = Path(__file__).resolve().parent
ROOT = HERE.parents[3]
sys.path.insert(0, str(HERE))
from Pixcel_Attribution import read_handmade, LABELS
from Automatic_Centering_Transformer import center_batch, center_shifts
from Input_Sensitivity import get_output

BASE = ROOT/"05_model_optimize"/"MLP"/"Normal"
NORMAL_MODEL=BASE/"results"/"Input_Sensitivity"/"normal_mlp_refit_stage3.joblib"
COM_MODEL=BASE/"results"/"Automatic_Centering"/"exp004_mass.joblib"
RECORDS=BASE/"results"/"Automatic_Centering"/"exp004_predictions.csv.gz"
NORMAL_SHA="9b54ba9c60e389575200d88ad0b93c47ec47f9c673eebeb755ff47e3ebe23b0a"
COM_SHA="3c1aac362d75ff5c4d9652188c1c4fd97524525f5409ec00d5a6c56fa3ad27ef"
COLS=("Normal_raw","Normal_CoM","CoMModel_raw","CoMModel_CoM")


def sha256(path:Path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()


def savecsv(path,rows):
    assert rows
    with path.open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)


def logits_for(net,raw):
    # raw pixels [0,255], exact existing pipeline's PixelScaler255 transformation.
    raw=np.asarray(raw,dtype=np.float32).reshape(-1,784)
    return get_output(type("Fake",(),{"named_steps":{"model":net}})(),raw)[0]


def true_vs_other_ig(net,raw,true,other,steps=128):
    """IG for logit(true) - logit(other), black baseline, fixed signed target."""
    x=np.asarray(raw,dtype=np.float64).reshape(784)/255.0
    w0,w1=(np.asarray(a,dtype=np.float64) for a in net.coefs_)
    b0=np.asarray(net.intercepts_[0],dtype=np.float64)
    b1=np.asarray(net.intercepts_[1],dtype=np.float64)
    assert len(net.coefs_)==2 and net.activation=="relu"
    cls=int(true); competitor=int(other)
    weight=w1[:,cls]-w1[:,competitor]
    aa=(np.arange(steps,dtype=np.float64)+.5)/steps
    z=(aa[:,None]*x[None,:])@w0+b0
    active=(z>0).astype(np.float64)
    grad=(active*weight[None,:])@w0.T
    ig=x*grad.mean(axis=0)
    def pairmargin(v):
        h=np.maximum(v@w0+b0,0)
        zz=h@w1+b1
        return float(zz[cls]-zz[competitor])
    delta=pairmargin(x)-pairmargin(np.zeros_like(x))
    residual=float(ig.sum()-delta)
    return ig.reshape(28,28),float(pairmargin(x)),residual


def metric(y,pred,mask,name,condition):
    truth=y[mask]; p=pred[mask]
    return {"group":name,"condition":condition,"n":int(mask.sum()),
            "correct":int((truth==p).sum()),
            "accuracy":float((truth==p).mean()),
            "true_digit_predictions":int((p<10).sum()),
            "true_letter_predictions":int((p>=10).sum())}


def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--out",type=Path,required=True)
    args=ap.parse_args()
    out=args.out.resolve();out.mkdir(parents=True,exist_ok=True)
    assert sha256(NORMAL_MODEL)==NORMAL_SHA
    assert sha256(COM_MODEL)==COM_SHA
    base_model=joblib.load(NORMAL_MODEL)
    com_model=joblib.load(COM_MODEL)
    assert base_model.named_steps["model"].get_params()==com_model.named_steps["model"].get_params()
    assert com_model.named_steps["centering"].method=="mass"
    x,y,writers,indices=read_handmade()
    assert len(y)==2560
    moved=center_batch(x,"mass")
    shifts=center_shifts(x,"mass")
    assert np.array_equal(com_model.named_steps["centering"].transform(x),moved)
    net_base=base_model.named_steps["model"]
    net_com=com_model.named_steps["model"]
    inputs={"Normal_raw":x, "Normal_CoM":moved,
            "CoMModel_raw":x, "CoMModel_CoM":moved}
    nets={"Normal_raw":net_base,"Normal_CoM":net_base,
          "CoMModel_raw":net_com,"CoMModel_CoM":net_com}
    output={name:logits_for(nets[name],inputs[name]) for name in COLS}
    pred={name:output[name].argmax(axis=1) for name in COLS}
    assert int((pred["Normal_raw"]==y).sum())==1609
    assert int((pred["CoMModel_CoM"]==y).sum())==1974
    assert np.array_equal(pred["Normal_raw"],base_model.predict(x))
    assert np.array_equal(pred["CoMModel_CoM"],com_model.predict(x))

    reference={}
    with gzip.open(RECORDS,"rt",encoding="utf-8-sig",newline="") as f:
        for row in csv.DictReader(f):
            if row["dataset"]=="Handmade":
                key=(int(row["writer"]),int(row["image_index"]))
                reference[key]=row
    assert len(reference)==2560
    for i,(writer,index,label) in enumerate(zip(writers,indices,y)):
        item=reference[(int(writer),int(index))]
        assert item["true"]==LABELS[int(label)]
        assert item["Normal_pred"]==LABELS[int(pred["Normal_raw"][i])]
        assert item["Center_of_Mass_pred"]==LABELS[int(pred["CoMModel_CoM"][i])]
    print("VERIFIED: 2560 previously published exact baseline/CoM predictions match",flush=True)

    groups=[("all",np.ones(len(y),dtype=bool)),
            ("digits_0-9",y<10),("letters_A-F",y>=10),
            ("true_5",y==5),("true_F",y==15),
            ("true_B",y==11),("true_8",y==8)]
    rows=[metric(y,pred[c],mask,g,c) for g,mask in groups for c in COLS]
    savecsv(out/"four_way_summary.csv",rows)
    print("FOUR-WAY CONTRAST: (fix weights for movement; fix input for fit)",flush=True)
    for group in ("all","digits_0-9","letters_A-F"):
        rs=[r for r in rows if r["group"]==group]
        print(f"{group}: "+", ".join(f"{r['condition']}={r['accuracy']:.6%}({r['correct']}/{r['n']})" for r in rs),flush=True)

    category_changes=[]
    for group,mask in groups:
        for a,b,label in [
            ("Normal_raw","Normal_CoM","Movement_same_normal_weights"),
            ("Normal_CoM","CoMModel_CoM","New_weights_same_centered_pixels"),
            ("Normal_raw","CoMModel_CoM","Actual_pipeline_delta"),
            ("CoMModel_raw","CoMModel_CoM","Movement_same_centered_weights_off_distribution"),
        ]:
            oldcorrect=(pred[a]==y)
            newcorrect=(pred[b]==y)
            m=mask
            category_changes.append({
                "group":group,"contrast":label,
                "source":a,"target":b,"n":int(m.sum()),
                "old_accuracy":float(oldcorrect[m].mean()),
                "new_accuracy":float(newcorrect[m].mean()),
                "delta_accuracy_pp":float(100*(newcorrect[m].mean()-oldcorrect[m].mean())),
                "corrected":int((m&~oldcorrect&newcorrect).sum()),
                "induced":int((m&oldcorrect&~newcorrect).sum()),
                "prediction_changes":int((m&(pred[a]!=pred[b])).sum()),
            })
    savecsv(out/"contrasts.csv",category_changes)

    # Actual 5->F correction, and true B that changes from correct B to wrong 8.
    wanted=[
        ("5_wrong_F_to_5",5,15,5),
        ("B_correct_to_wrong_8",11,11,8),
    ]
    cases=[]
    for scenario,true,before,after in wanted:
        hits=np.flatnonzero((y==true)&(pred["Normal_raw"]==before)&
                            (pred["CoMModel_CoM"]==after))
        assert len(hits)>0,(scenario,true,before,after)
        print(f"SCENARIO {scenario}: count={len(hits)}",flush=True)
        # Intentionally sample diverse writers when possible and include meaningful movements.
        picked=[]
        for writer in (41,42,43,44):
            matches=[int(i) for i in hits if writers[i]==writer]
            if matches:
                matches.sort(key=lambda i:(-np.linalg.norm(shifts[i]),int(indices[i])))
                picked.append(matches[0])
        for i in picked[:4]:
            cases.append((scenario,i,true,15 if true==5 else 8))
    print("SELECTED CASES:",[(sc,int(writers[i]),int(indices[i])) for sc,i,_,_ in cases],flush=True)

    igrows=[]
    gridfigs=[]
    for scenario,i,true,other in cases:
        raw=x[i].reshape(28,28);centered=moved[i].reshape(28,28)
        signed=[]
        for name,img,net in [
            ("Normal_raw",raw,net_base),
            ("Normal_CoM",centered,net_base),
            ("CoMModel_CoM",centered,net_com)]:
            ig,margin,resid=true_vs_other_ig(net,img,true,other)
            signed.append((name,ig,margin,resid))
            igrows.append({
                "scenario":scenario,
                "writer":int(writers[i]),"index":int(indices[i]),
                "true":LABELS[true],"competitor":LABELS[other],
                "model_condition":name,
                "prediction":LABELS[int(pred[name][i])],
                "true_minus_competitor_margin":margin,
                "IG_signed_sum":float(ig.sum()),
                "IG_positive_sum":float(np.clip(ig,0,None).sum()),
                "IG_negative_sum":float(np.clip(ig,None,0).sum()),
                "IG_max_abs":float(np.max(np.abs(ig))),
                "IG_completeness_residual":resid,
                "shift_x":int(shifts[i,0]),"shift_y":int(shifts[i,1])
            })
        gridfigs.append((scenario,i,raw,centered,signed))

    savecsv(out/"selected_ig_diagnostics.csv",igrows)

    # Image grids: one 5->F corrected and one B->8 induced per writer when available.
    for scenario in ("5_wrong_F_to_5","B_correct_to_wrong_8"):
        chosen=[x for x in gridfigs if x[0]==scenario]
        if not chosen:
            continue
        n=len(chosen)
        fig,axes=plt.subplots(n,5,figsize=(16,3.45*n),squeeze=False)
        for k,(_,i,raw,centered,signed) in enumerate(chosen):
            amps=max(float(np.abs(z[1]).max()) for z in signed)
            for col,view in ((0,raw),(1,centered)):
                ax=axes[k,col]
                ax.imshow(view,cmap="gray",vmin=0,vmax=255,interpolation="nearest")
                ax.set_xticks([]);ax.set_yticks([])
            for col,(name,ig,margin,resid) in enumerate(signed,start=2):
                ax=axes[k,col]
                ax.imshow(ig,cmap="RdBu_r",vmin=-amps,vmax=amps,
                          interpolation="nearest")
                ax.set_xticks([]);ax.set_yticks([])
                ax.set_title(f"{name}\npred {LABELS[int(pred[name][i])]}, "
                             f"margin {margin:+.2f}",fontsize=9)
            axes[k,0].set_title(f"Original W{writers[i]} #{indices[i]}\n"
                                f"pred {LABELS[int(pred['Normal_raw'][i])]}",fontsize=10)
            axes[k,1].set_title(f"CoM shifted dx={shifts[i,0]:+d} dy={shifts[i,1]:+d}\n"
                                f"pred {LABELS[int(pred['CoMModel_CoM'][i])]}",fontsize=10)
        true="5" if scenario.startswith("5") else "B"
        other="F" if true=="5" else "8"
        fig.suptitle(f"{scenario}: Signed IG for logit({true})−logit({other})\n"
                     "Red favors true label; blue favors competitor. "
                     "First compare col 3↔4 (movement), then col 4↔5 (training).",
                     fontsize=12)
        fig.tight_layout(rect=[0,0,1,.94])
        fig.savefig(out/f"IG_{scenario}.png",dpi=150)
        plt.close(fig)

    # 2x2 controlled prediction CSV; no labels used for interventions.
    perimage=[]
    for i in range(len(y)):
        record={"writer":int(writers[i]),"index":int(indices[i]),
                "true":LABELS[int(y[i])],
                "dx":int(shifts[i,0]),"dy":int(shifts[i,1])}
        for c in COLS:
            record[f"{c}_pred"]=LABELS[int(pred[c][i])]
            record[f"{c}_correct"]=int(pred[c][i]==y[i])
        perimage.append(record)
    savecsv(out/"four_way_predictions.csv",perimage)

    # True-5 and true-B class margin aggregate over all 160 true-class samples each.
    margins=[]
    for actual,other in ((5,15),(11,8)):
        ix=np.flatnonzero(y==actual)
        for name in COLS:
            val=output[name][ix,actual]-output[name][ix,other]
            margins.append({
                "true":LABELS[actual],"competitor":LABELS[other],
                "condition":name,"n":len(ix),
                "positive_pairwise_margin_fraction":float((val>0).mean()),
                "mean_true_minus_other_logit_margin":float(val.mean()),
                "median_true_minus_other_logit_margin":float(np.median(val)),
                "p10_margin":float(np.quantile(val,.1)),
                "p90_margin":float(np.quantile(val,.9)),
            })
    savecsv(out/"pairwise_margins.csv",margins)

    meta={"study":"EXP유004 CoM pixel decision diagnostic","mode":"review-only",
          "handmade_n":2560,"official_test_read":False,"professor_test_read":False,
          "model_fit_performed":False,"new_preprocessing_parameter_search":False,
          "models_sha256":{"Normal":NORMAL_SHA,"CoM":COM_SHA},
          "fixed_target_comparisons":["5_minus_F","B_minus_8"],
          "ig_reference":"all-zero input",
          "ig_approximation":"128 midpoint",
          "ig_count":len(igrows),
          "selected_examples":len(cases),
          "versions":{"python":platform.python_version(),"numpy":np.__version__,
                      "scikit_learn":sklearn.__version__}}
    (out/"metadata.json").write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding="utf-8")
    overview=["# EXP유004 follow-up — CoM decision diagnostic (review only)",
              "",
              "- The same 2,560 Handmade files (not independent unseen-writer Test).",
              "- No new training, no Test, no hyperparameter tuning, no model deployment.",
              "- Compare Normal_raw vs Normal_CoM: input shift with identical weights.",
              "- Compare Normal_CoM vs CoMModel_CoM: weights difference on identical centered images.",
              "- CoMModel_raw is **off the new model training distribution**, diagnostic only.",
              "- IG uses signed margin true-class logit minus fixed rival class (5-F or B-8).",
              "- Red IG region supports true class *relative to rival*, blue supports rival.",
              "- A pixel IG map is not evidence of human-like shape understanding or proof of causality.",
              "",
              "| Group | Normal/raw | Normal/CoM shift | CoM-model/raw (diagnostic) | CoM-model/CoM |",
              "|---|---:|---:|---:|---:|"]
    for group in ("all","digits_0-9","letters_A-F","true_5","true_B"):
        rr=[next(r for r in rows if r["group"]==group and r["condition"]==name) for name in COLS]
        overview.append(f"| {group} | "+" | ".join(f"{r['accuracy']:.2%}" for r in rr)+" |")
    overview+=["","## Paired contrasts (causal controlled manipulations on the same samples)"]
    for group in ("digits_0-9","letters_A-F"):
        overview.append(f"### {group}")
        overview.append("| Comparison | Accuracy delta (pp) | Corrected | New errors |")
        overview.append("|---|---:|---:|---:|")
        for r in category_changes:
            if r["group"]==group and r["contrast"] in [
                "Movement_same_normal_weights","New_weights_same_centered_pixels","Actual_pipeline_delta"]:
                overview.append(f"| {r['contrast']} | {r['delta_accuracy_pp']:+.3f} | "
                                f"{r['corrected']} | {r['induced']} |")
    overview+=["","## Selected pairs",""]
    for scenario,_,_,_ in cases:
        pass
    overview.append(f"- True 5, baseline F → CoM model 5: "
                    f"{sum(1 for s,_,_,_ in cases if s=='5_wrong_F_to_5')} illustrated examples.")
    overview.append(f"- True B, baseline B → CoM model 8: "
                    f"{sum(1 for s,_,_,_ in cases if s=='B_correct_to_wrong_8')} illustrated examples.")
    overview+=["","## Files",
               "- four_way_summary.csv, contrasts.csv: exact paired group results",
               "- four_way_predictions.csv: same-image 2×2 controls for all 2,560",
               "- pairwise_margins.csv: true 5-vs-F and B-vs-8 logit margins",
               "- selected_ig_diagnostics.csv: per-example signed IG and completeness",
               "- IG_5_wrong_F_to_5.png; IG_B_correct_to_wrong_8.png: actual image/IG grids",
               "- metadata.json: inputs, checkpoint hashes, no-Test confirmation"]
    (out/"SUMMARY.md").write_text("\n".join(overview)+"\n",encoding="utf-8")
    print("\n".join(overview[:35]),flush=True)
    print("DIAGNOSTIC COMPLETE; generated files:",sorted(p.name for p in out.iterdir()),flush=True)


if __name__=="__main__":
    main()


## CoM_Translation_Augmentation_Experiment

원본: [05_model_optimize/MLP/Normal/scripts/CoM_Translation_Augmentation_Experiment.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/CoM_Translation_Augmentation_Experiment.py)  
Git blob: `68bd0266b898cba4bbb0f400d04d5b5ce34daf8d`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

EXP유005: CoM-centered MLP with controlled 1px translation augmentation.

Experiment A: the verified EXP유004 CoM MLP (frozen, not retrained).
Experiment B: same 76,800 official train images after CoM centering, repeated
              5x (384,000 identical images) to control sample/optimizer exposure.
Experiment C: the same CoM centered images each appearing five times:
              [original, left 1px, right 1px, up 1px, down 1px].
Evaluation: CoM centering ONLY on official Validation and Handma

CLI 인자 (정적 추출): `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/CoM_Translation_Augmentation_Experiment.py
#!/usr/bin/env python3
"""EXP유005: CoM-centered MLP with controlled 1px translation augmentation.

Experiment A: the verified EXP유004 CoM MLP (frozen, not retrained).
Experiment B: same 76,800 official train images after CoM centering, repeated
              5x (384,000 identical images) to control sample/optimizer exposure.
Experiment C: the same CoM centered images each appearing five times:
              [original, left 1px, right 1px, up 1px, down 1px].
Evaluation: CoM centering ONLY on official Validation and Handmade; never
augmentation on Validation/Handmade and never reads any official Test split.
No class-dependent preprocessing and no Handmade fitting. Output REVIEW ONLY;
do not update repo or auto-select candidate without an explicit user decision.
"""
from __future__ import annotations

import argparse
import csv
import gc
import gzip
import hashlib
import json
import platform
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import scipy
import sklearn
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline

HERE = Path(__file__).resolve().parent
ROOT = HERE.parents[3]
sys.path.insert(0, str(HERE))

from Pixcel_Attribution import (
    fixed_split, DEFAULT_DATA, read_handmade, LABELS, build_baseline,
)
from Automatic_Centering_Transformer import (
    AutomaticCentering, center_batch, test_centering, THRESHOLD, TARGET,
)
from Input_Sensitivity import transform, Variant
from pixel_transform import PixelScaler255

DIR = ROOT / "05_model_optimize" / "MLP" / "Normal"
REFERENCE_MODEL = DIR / "results" / "Automatic_Centering" / "exp004_mass.joblib"
REFERENCE_PREDICTIONS = DIR / "results" / "Automatic_Centering" / "exp004_predictions.csv.gz"
REFERENCE_MODEL_SHA = "3c1aac362d75ff5c4d9652188c1c4fd97524525f5409ec00d5a6c56fa3ad27ef"
REFERENCE_DATA_SHA = "0e9570d0eb04c0c1da99d043b85bfdab7f3b639eebe4b10335b05b57ad671879"
CONDITIONS = ("EXP004_CoM", "EXP005_repeat5_control", "EXP005_shift1_augmented")
DIRECTIONS = (
    ("original", 0, 0),
    ("left1", -1, 0),
    ("right1", 1, 0),
    ("up1", 0, -1),
    ("down1", 0, 1),
)
PAIR_ERRORS = (("5", "F"), ("4", "F"), ("B", "8"), ("8", "B"),
               ("0", "D"), ("D", "0"), ("9", "4"), ("E", "C"))


def checksum(path: Path) -> str:
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()


def write_csv(path: Path, records: list[dict]) -> None:
    if not records:
        raise ValueError("No rows for " + str(path))
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)


def shift_into(source: np.ndarray, target: np.ndarray, dx: int, dy: int) -> None:
    """Move pre-centered pixel arrays without wrapping, interpolation or recentering."""
    assert source.shape == target.shape and source.ndim == 3 and source.shape[1:] == (28, 28)
    assert abs(dx) <= 1 and abs(dy) <= 1
    assert not np.shares_memory(source, target)
    target.fill(0)
    sy0, sy1 = max(0, -dy), min(28, 28 - dy)
    sx0, sx1 = max(0, -dx), min(28, 28 - dx)
    if sy0 < sy1 and sx0 < sx1:
        target[:, sy0 + dy:sy1 + dy, sx0 + dx:sx1 + dx] = source[:, sy0:sy1, sx0:sx1]


def test_augmentation():
    test_centering()
    x = np.zeros((3, 28, 28), dtype=np.float32)
    x[0, 14, 14], x[1, 0, 0], x[2, 27, 27] = 255, 190, 110
    for label, dx, dy in DIRECTIONS:
        actual = np.empty_like(x)
        shift_into(x, actual, dx, dy)
        expected = transform(x.reshape(3, 784),
                             Variant(label, "original", None) if label == "original" else
                             Variant(label, "shift", (dx, dy)))
        assert np.array_equal(actual.reshape(3, 784), expected), label
        assert np.all(x[[0, 1, 2], [14, 0, 27], [14, 0, 27]] == [255, 190, 110])
    print("EXP005 SELF-TEST PASSED: ±1 px zero-filled translations exactly match EXP유002", flush=True)


def build_training(centered: np.ndarray, kind: str) -> tuple[np.ndarray, dict]:
    """Construct 5 blocks in fixed order for both conditions; convert to /255 in-place."""
    n = len(centered)
    assert n == 76800 and centered.shape == (n, 784)
    raw = np.asarray(centered, dtype=np.float32).reshape(n, 28, 28)
    result = np.empty((5 * n, 784), dtype=np.float32)
    mass_before = raw.sum(axis=(1, 2), dtype=np.float64)
    clip = {}
    for j, (name, dx, dy) in enumerate(DIRECTIONS):
        out = result[j*n:(j+1)*n].reshape(n, 28, 28)
        if kind == "repeat5":
            out[:] = raw
        else:
            assert kind == "shift1"
            shift_into(raw, out, dx, dy)
        mass_after = out.sum(axis=(1, 2), dtype=np.float64)
        lost = np.maximum(0.0, mass_before - mass_after)
        clip[name] = {
            "images_with_ink_loss": int(np.count_nonzero(lost > 1e-2)),
            "mean_relative_ink_loss": float(np.mean(lost / np.maximum(mass_before, 1))),
        }
    if kind == "repeat5":
        for j in range(1, 5):
            assert np.array_equal(result[0:n], result[j*n:(j+1)*n])
    else:
        assert np.array_equal(result[:n], centered), "CoM must precede translation"
        # One-pixel augmentation should actually change some original training images.
        assert any(not np.array_equal(result[:n], result[j*n:(j+1)*n]) for j in range(1, 5))
    result /= np.float32(255.0)
    assert result.dtype == np.float32 and np.isfinite(result).all()
    assert result.min() >= 0 and result.max() <= 1
    return result, clip


def metric_row(dataset, condition, group_type, group, y, p, reference):
    n = len(y)
    good = p == y
    old = reference == y
    return {
        "dataset": dataset, "condition": condition, "group_type": group_type,
        "group": str(group), "n": n, "correct": int(good.sum()),
        "accuracy": float(good.mean()),
        "macro_f1_present_classes": float(f1_score(y, p, average="macro")),
        "corrected_vs_EXP004_CoM": int((~old & good).sum()),
        "induced_vs_EXP004_CoM": int((old & ~good).sum()),
        "changed_predictions_vs_EXP004_CoM": int((p != reference).sum()),
        "delta_accuracy_pp_vs_EXP004_CoM": float((good.mean() - old.mean()) * 100),
    }


def group_rows(dataset, y, writer, all_preds):
    records = []
    ref = all_preds["EXP004_CoM"]
    for condition, pred in all_preds.items():
        groups = [("all", "all", np.ones(len(y), dtype=bool))]
        if dataset == "Handmade":
            groups.extend([("category", "digits", y < 10),
                           ("category", "letters", y >= 10)])
            for w in (41, 42, 43, 44):
                groups.append(("writer", w, writer == w))
                groups.append(("writer_category", f"{w}_digits", (writer == w) & (y < 10)))
                groups.append(("writer_category", f"{w}_letters", (writer == w) & (y >= 10)))
        for cls in range(16):
            groups.append(("class", LABELS[cls], y == cls))
        for typ, name, mask in groups:
            records.append(metric_row(dataset, condition, typ, name,
                                      y[mask], pred[mask], ref[mask]))
    return records


def paired_transition(group, y, pred_before, pred_after, condition_a, condition_b):
    ca = pred_before == y
    cb = pred_after == y
    return {
        "group": group,
        "from": condition_a, "to": condition_b, "n": len(y),
        "accuracy_before": float(ca.mean()),
        "accuracy_after": float(cb.mean()),
        "delta_accuracy_pp": float((cb.mean() - ca.mean()) * 100),
        "corrected": int((~ca & cb).sum()),
        "new_errors": int((ca & ~cb).sum()),
        "changed_predictions": int((pred_before != pred_after).sum()),
    }


def compare_transitions(y, writer, preds):
    rows = []
    groups = [("all", np.ones(len(y), dtype=bool)),
              ("digits", y < 10), ("letters", y >= 10)]
    for w in (41, 42, 43, 44):
        groups.append((f"writer_{w}", writer == w))
    for a, b in (
        ("EXP004_CoM", "EXP005_repeat5_control"),
        ("EXP004_CoM", "EXP005_shift1_augmented"),
        ("EXP005_repeat5_control", "EXP005_shift1_augmented"),
    ):
        for name,mask in groups:
            rows.append(paired_transition(name, y[mask], preds[a][mask],
                                          preds[b][mask], a, b))
    return rows


def verify_exp004(yva, yhm, whm, ihm, predv, predh):
    assert checksum(REFERENCE_MODEL) == REFERENCE_MODEL_SHA
    assert (predv == yva).sum() == 18608
    assert (predh == yhm).sum() == 1974
    exact = {"OfficialVal": [], "Handmade": {}}
    with gzip.open(REFERENCE_PREDICTIONS, "rt", encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            if row["dataset"] == "OfficialVal":
                exact["OfficialVal"].append((row["true"], row["Center_of_Mass_pred"]))
            elif row["dataset"] == "Handmade":
                exact["Handmade"][(int(row["writer"]), int(row["image_index"]))] = (
                    row["true"], row["Center_of_Mass_pred"])
            else:
                raise ValueError("Unexpected dataset in published EXP004")
    assert len(exact["OfficialVal"]) == 19200
    assert len(exact["Handmade"]) == 2560
    for i, (true, pred) in enumerate(exact["OfficialVal"]):
        assert true == LABELS[int(yva[i])] and pred == LABELS[int(predv[i])]
    for i,(w,ix) in enumerate(zip(whm,ihm)):
        true,pred=exact["Handmade"][(int(w),int(ix))]
        assert true==LABELS[int(yhm[i])] and pred==LABELS[int(predh[i])]
    print("EXP004 REPRODUCED: Official Val 18608/19200; Handmade 1974/2560; all 21760 predictions exact",flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", type=Path, required=True)
    args = ap.parse_args()
    out = args.out.resolve()
    out.mkdir(parents=True, exist_ok=True)
    start = time.monotonic()

    test_augmentation()
    xtr, ytr, xva, yva, input_sha = fixed_split(DEFAULT_DATA)
    xhm, yhm, whm, ihm = read_handmade()
    assert (len(xtr),len(xva),len(xhm)) == (76800,19200,2560)
    assert input_sha == REFERENCE_DATA_SHA
    assert sorted(np.unique(ytr)) == list(range(16))
    assert len(DIRECTIONS) == 5
    print("DATA VERIFIED: 76800 Official Train / 19200 Official Val / 2560 Handmade. No Test read.",flush=True)

    xtr_com = center_batch(xtr,"mass")
    # One consistent inference preprocessing, NO evaluation augmentation.
    xva_com01 = center_batch(xva,"mass") / np.float32(255)
    xhm_com01 = center_batch(xhm,"mass") / np.float32(255)
    # The original EXP004 centering Transformer remains in the final pipeline.
    original_model = joblib.load(REFERENCE_MODEL)
    assert isinstance(original_model.named_steps["centering"], AutomaticCentering)
    assert original_model.named_steps["centering"].method == "mass"
    estimator_params = build_baseline()[0].get_params()
    assert original_model.named_steps["model"].get_params() == estimator_params
    pred = {
        "OfficialVal": {"EXP004_CoM": original_model.predict(xva)},
        "Handmade": {"EXP004_CoM": original_model.predict(xhm)},
    }
    assert np.array_equal(pred["OfficialVal"]["EXP004_CoM"],
                          original_model.named_steps["model"].predict(xva_com01))
    assert np.array_equal(pred["Handmade"]["EXP004_CoM"],
                          original_model.named_steps["model"].predict(xhm_com01))
    verify_exp004(yva,yhm,whm,ihm,pred["OfficialVal"]["EXP004_CoM"],
                   pred["Handmade"]["EXP004_CoM"])

    y_train = np.tile(ytr,5)
    assert len(y_train)==384000
    durations, warnings_by_condition, stopping, clipping = {}, {}, {}, {}
    for condition, kind, checkpoint in (
        ("EXP005_repeat5_control", "repeat5", "exp005_repeat5_control.joblib"),
        ("EXP005_shift1_augmented", "shift1", "exp005_shift1_augmented.joblib"),
    ):
        print(f"TRAIN {condition}: constructing exactly 384000 centered / shifted inputs...",flush=True)
        t0=time.monotonic()
        training, clipping[kind] = build_training(xtr_com,kind)
        print(f"TRAIN {condition}: data ready, shape={training.shape}, "
              f"dtype={training.dtype}, MB={training.nbytes/1e6:.1f}",flush=True)
        estimator, uses_scaler = build_baseline()
        assert uses_scaler and estimator.get_params()==estimator_params
        with warnings.catch_warnings(record=True) as records:
            warnings.simplefilter("always")
            estimator.fit(training, y_train)
        durations[condition] = float(time.monotonic()-t0)
        warnings_by_condition[condition] = [
            f"{w.category.__name__}: {w.message}" for w in records]
        stopping[condition] = {
            "n_iter": int(estimator.n_iter_), "loss": float(estimator.loss_),
            "loss_curve": [float(t) for t in estimator.loss_curve_],
            "n_epochs_max": int(estimator.max_iter),
            "trained_examples_per_epoch":384000,
            "training_parameter_count":sum(int(v.size) for v in estimator.coefs_+estimator.intercepts_),
        }
        assert estimator.get_params()==estimator_params
        del training
        gc.collect()

        # Inference pipeline (CoM ONLY) is identical in structure to EXP004.
        pipe=Pipeline([("centering",AutomaticCentering(method="mass")),
                       ("scale_0_1",PixelScaler255()),("model",estimator)])
        pva=estimator.predict(xva_com01)
        phm=estimator.predict(xhm_com01)
        assert np.array_equal(pva,pipe.predict(xva))
        assert np.array_equal(phm,pipe.predict(xhm))
        pred["OfficialVal"][condition]=pva
        pred["Handmade"][condition]=phm
        joblib.dump(pipe,out/checkpoint,compress=3)
        print(f"TRAINED {condition}: {durations[condition]:.1f}s, iterations={estimator.n_iter_}, "
              f"Official Val={accuracy_score(yva,pva):.6%}, "
              f"Handmade={accuracy_score(yhm,phm):.6%}, "
              f"Handmade digits={(phm[yhm<10]==yhm[yhm<10]).mean():.6%}, "
              f"letters={(phm[yhm>=10]==yhm[yhm>=10]).mean():.6%}",flush=True)
        del estimator, pipe
        gc.collect()

    # Official Val and Handmade class/writer/category metrics
    metrics=group_rows("OfficialVal",yva,np.full(len(yva),-1,dtype=np.int32),
                       pred["OfficialVal"])
    metrics+=group_rows("Handmade",yhm,whm,pred["Handmade"])
    write_csv(out/"exp005_group_metrics.csv",metrics)
    transitions=compare_transitions(yhm,whm,pred["Handmade"])
    write_csv(out/"exp005_paired_transitions.csv",transitions)

    # Per-image predictions and binary correct flags preserve exact source labels.
    rows=[]
    for i,(w,ix,y) in enumerate(zip(whm,ihm,yhm)):
        rec={"writer":int(w),"image_index":int(ix),"true":LABELS[int(y)]}
        for condition in CONDITIONS:
            pi=int(pred["Handmade"][condition][i])
            rec[condition+"_prediction"]=LABELS[pi]
            rec[condition+"_correct"]=int(pi==int(y))
        rows.append(rec)
    with gzip.open(out/"exp005_handmade_predictions.csv.gz","wt",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)

    confusions=[]
    for domain,y in (("OfficialVal",yva),("Handmade",yhm)):
        for condition in CONDITIONS:
            cm=confusion_matrix(y,pred[domain][condition],labels=np.arange(16))
            assert cm.sum()==len(y)
            for i,j in np.ndindex((16,16)):
                confusions.append({"dataset":domain,"condition":condition,
                                   "true":LABELS[i],"predicted":LABELS[j],
                                   "count":int(cm[i,j])})
    write_csv(out/"exp005_confusion_matrices.csv",confusions)
    pairs=[]
    for domain,y in (("OfficialVal",yva),("Handmade",yhm)):
        for condition in CONDITIONS:
            for a,b in PAIR_ERRORS:
                pairs.append({"dataset":domain,"condition":condition,
                              "true":a,"predicted":b,
                              "errors":int(((y==LABELS.index(a)) &
                               (pred[domain][condition]==LABELS.index(b))).sum())})
    write_csv(out/"exp005_confusion_pairs.csv",pairs)

    loss=[]
    for condition in ("EXP005_repeat5_control","EXP005_shift1_augmented"):
        for epoch,l in enumerate(stopping[condition]["loss_curve"],start=1):
            loss.append({"condition":condition,"epoch":epoch,"training_loss":l})
    write_csv(out/"exp005_learning_curves.csv",loss)

    def stat(dataset, condition, group):
        return next(r for r in metrics if r["dataset"]==dataset and
                    r["condition"]==condition and
                    ((r["group_type"]=="all" and group=="all") or
                     (r["group_type"]=="category" and r["group"]==group)))
    overview=[
        "# EXP유005 — CoM 정렬 후 ±1px 위치 증강 (REVIEW ONLY, 실측)",
        "",
        "연구 질문: CoM 위치 정렬은 유지하면서 MLP의 절대 픽셀 위치 의존성을 줄이면 "
        "기존 문자 정확도 66.98%를 개선하고 숫자 83.19%를 유지할 수 있는가?",
        "",
        "**비교 A:** 기존 EXP유004 CoM frozen 모델, 공식 76,800장 Training으로 학습한 후보 그대로.",
        "**비교 B:** 같은 CoM 정렬 공식 Train 76,800장을 5회 정확히 반복해 384,000장으로 학습.",
        "**비교 C:** CoM 정렬 공식 Train 76,800장에 원본/좌/우/상/하 1px 변형 5종을 적용해 384,000장으로 학습.",
        "비교 B/C의 샘플 수·순서·레이블·MLP 구조·random_state=42는 동일. "
        "모델은 두 조건에서 각각 새로 학습하므로 총 epoch/optimizer update 수는 "
        "실제 수렴 경로에 따라 달라질 수 있다.",
        "",
        "**중요:** 위치 증강은 **CoM 정렬 이후 Training에만** 적용. "
        "Official Validation과 Handmade에는 CoM 정렬만 적용.",
        "**Test 사용 없음:** 공식 Original Test, 교수님 Test, "
        "Handmade fit/hyperparameter selection 모두 미실시.",
        "",
        "| 평가 | EXP유004 CoM | EXP유005 반복5 대조군 | EXP유005 CoM+±1px |",
        "|---|---:|---:|---:|",
    ]
    for domain,group in (("OfficialVal","all"),("Handmade","all"),
                         ("Handmade","digits"),("Handmade","letters")):
        accs=[stat(domain,c,group)["accuracy"] for c in CONDITIONS]
        overview.append(f"| {domain} {group} | "+" | ".join(f"{v:.4%}" for v in accs)+" |")
    overview.extend([
        "",
        "## 정확도 개선 vs 재발생 오류 (Handmade paired)",
        "| Group | Comparison | delta (pp) | Corrected | New errors |",
        "|---|---|---:|---:|---:|"])
    for row in transitions:
        if row["group"] in ("all","digits","letters"):
            overview.append(f"| {row['group']} | {row['from']}→{row['to']} | "
                            f"{row['delta_accuracy_pp']:+.3f} | "
                            f"{row['corrected']} | {row['new_errors']} |")
    overview.extend(["","## 학습·평가 조건",""])
    for condition in ("EXP005_repeat5_control","EXP005_shift1_augmented"):
        s=stopping[condition]
        overview.append(f"- {condition}: {durations[condition]:.1f}초, "
                        f"epochs={s['n_iter']}, last training loss={s['loss']:.6g}, "
                        f"warnings={warnings_by_condition[condition]}.")
    overview.extend([
        "",
        "**해석 주의:** 4명의 Hand-made는 이미 여러 번 분석된 개발용 데이터다. "
        "독립된 신규 작성자에 대한 일반화 성능은 아직 확인되지 않았다. "
        "공식 Test나 교수님 Test를 중간 비교에 사용하지 않는다.",
        "",
        "원본 검증 파일: exp005_group_metrics.csv, exp005_paired_transitions.csv, "
        "exp005_confusion_matrices.csv, exp005_confusion_pairs.csv, "
        "exp005_handmade_predictions.csv.gz, exp005_learning_curves.csv, "
        "exp005_metadata.json, 두 후보 joblib.",
        "",
        "**중간 판단은 배포/최종 모델 채택이 아님.**"])
    (out/"SUMMARY.md").write_text("\n".join(overview)+"\n",encoding="utf-8")
    info={
        "study":"EXP유005 CoM_Translation_Augmentation",
        "status":"review-only; no official results/logs publication",
        "official_npz_sha256":input_sha,
        "exp004_com_checkpoint_sha256":REFERENCE_MODEL_SHA,
        "official_train":len(ytr), "official_validation":len(yva),
        "handmade":len(yhm), "handmade_writers":[41,42,43,44],
        "handmade_used_in_fit":False, "handmade_used_for_preprocessing_selection":False,
        "official_original_test_read":False, "professor_test_read":False,
        "baseline_exp004_21760_pred_verified":True,
        "training_conditions":{
            "control":"CoM aligned identical 76,800 source images repeated 5 times",
            "candidate":"CoM aligned first; same original plus 1px L/R/U/D",
            "train_size":384000, "labels_repeated":5,
            "order_of_blocks":[a for a,_,_ in DIRECTIONS],
        },
        "evaluation_preprocess":"AutomaticCentering(method=mass), PixelScaler255 only (no shift)",
        "centering":{"threshold_0_to_255":THRESHOLD,
                     "target_xy":list(TARGET),
                     "translation":"np.rint integer offset, zero-fill, no wrap, no interpolation"},
        "training_model_params":{k:repr(v) for k,v in estimator_params.items()},
        "model_training_seconds":durations,
        "model_warnings":warnings_by_condition,
        "epochs_loss_parameters":stopping,
        "translation_ink_clipping":clipping,
        "software":{"python":platform.python_version(),"numpy":np.__version__,
                    "scipy":scipy.__version__,"sklearn":sklearn.__version__},
        "elapsed_seconds":float(time.monotonic()-start),
    }
    (out/"exp005_metadata.json").write_text(
        json.dumps(info,ensure_ascii=False,indent=2),encoding="utf-8")
    print("\n".join(overview),flush=True)
    print("EXP005 COMPLETE; all files:",sorted(q.name for q in out.iterdir()),flush=True)


if __name__ == "__main__":
    main()


## exp006_size_com_comparison

원본: [05_model_optimize/MLP/Normal/scripts/exp006_size_com_comparison.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/exp006_size_com_comparison.py)  
Git blob: `808dcf7e34164dd86890f846521bd481acc2eb70`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

Normal Hypothesis B, EXP006: exact Combined Size/CoM comparison.

Uses only OFFICIAL TRAIN data for fitting (76,800 images); evaluates on
OFFICIAL VALIDATION (19,200) and existing 4-author Hand-made developer data
(2,560). Never accesses Original Test or professor Test. No GitHub writes.

CLI 인자 (정적 추출): `--method`, `--seed`, `--out`, `--aggregate`, `--inputs`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/exp006_size_com_comparison.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Normal Hypothesis B, EXP006: exact Combined Size/CoM comparison.

Uses only OFFICIAL TRAIN data for fitting (76,800 images); evaluates on
OFFICIAL VALIDATION (19,200) and existing 4-author Hand-made developer data
(2,560). Never accesses Original Test or professor Test. No GitHub writes.
"""
from __future__ import annotations

import argparse
import csv
import gzip
import hashlib
import json
import os
import platform
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import scipy
import sklearn
from joblib import dump
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

ROOT = Path(__file__).resolve().parents[4]
CMB = ROOT / "05_model_optimize" / "MLP" / "Combined" / "scripts"
NORM = ROOT / "05_model_optimize" / "MLP" / "Normal" / "scripts"
sys.path.insert(0, str(CMB))
sys.path.insert(0, str(NORM))

# Reuse the exact pixel processing used in Combined EXP-CMB-004.
from combined_mlp_writer_cv import load_official, load_writers, PixelScaler255
from exp_cmb004_geometry_normalization import image_geometry, apply_size
from Automatic_Centering_Transformer import center_batch

WRITERS = (41, 42, 43, 44)
LABELS = "0123456789ABCDEF"
METHODS = ("com", "size", "size_com")
PAIRS = (("B","8"), ("8","B"), ("D","0"), ("0","D"),
         ("5","F"), ("F","5"))


def save_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        if not rows:
            raise ValueError("Cannot infer fields for empty CSV")
        fieldnames = list(rows[0])
    with Path(path).open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def process(x, method, target):
    if method == "com":
        a = center_batch(x, "mass")
    elif method == "size":
        a, _ = apply_size(x, target)
    elif method == "size_com":
        a, _ = apply_size(x, target)
        a = center_batch(a, "mass")
    else:
        raise ValueError(method)
    a = np.asarray(a, dtype=np.float32).reshape(-1, 784)
    assert np.isfinite(a).all() and 0 <= a.min() <= a.max() <= 255
    return a


def score(y, pred):
    return {
        "n": int(len(y)),
        "correct": int(np.count_nonzero(y == pred)),
        "accuracy": float(accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, labels=np.unique(y),
                                   average="macro", zero_division=0)),
    }


def evaluate_group(y, pred, writer_ids=None):
    output = {"all": score(y, pred),
              "digits": score(y[y < 10], pred[y < 10]),
              "letters": score(y[y >= 10], pred[y >= 10])}
    if writer_ids is not None:
        for w in WRITERS:
            ids = writer_ids == w
            output["writer_" + str(w)] = score(y[ids], pred[ids])
    return output


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for data in iter(lambda: f.read(1024 * 1024), b""):
            h.update(data)
    return h.hexdigest()


def train(method, seed, out):
    out = Path(out)
    if out.exists():
        raise RuntimeError("Refusing to overwrite existing results: " + str(out))
    out.mkdir(parents=True)
    data_path = ROOT / "02_data" / "datasets" / "MNIST_EMNIST_official_split_112000.npz"
    official = load_official(data_path)
    hm = load_writers(ROOT / "02_data" / "Hand_made")
    assert official["X_train"].shape == (76800, 784)
    assert official["X_val"].shape == (19200, 784)
    # No information from validation or handwriting enters this target.
    target = int(np.median(image_geometry(official["X_train"])["long_side"]))
    if target != 20:
        raise AssertionError("Combined target no longer reproducible: " + str(target))
    print("OFFICIAL TRAIN ONLY: fitted target long side =", target, flush=True)

    train_x = process(official["X_train"], method, target)
    val_x = process(official["X_val"], method, target)
    hm_raw = np.concatenate([hm[w]["X"] for w in WRITERS], axis=0)
    hm_y = np.concatenate([hm[w]["y"] for w in WRITERS], axis=0)
    writers = np.repeat(np.array(WRITERS), 640)
    hm_x = process(hm_raw, method, target)

    estimator = Pipeline([
        ("scale_0_1", PixelScaler255()),
        ("model", MLPClassifier(random_state=seed)),
    ])
    t0 = time.monotonic()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        estimator.fit(train_x, official["y_train"])
    fit_seconds = time.monotonic() - t0
    pred_val = estimator.predict(val_x).astype(int)
    pred_hm = estimator.predict(hm_x).astype(int)
    official_metrics = evaluate_group(official["y_val"], pred_val)
    handmade_metrics = evaluate_group(hm_y, pred_hm, writers)

    # Cross-run reproduction is informative, but the actual controlled comparison
    # is within THIS run: all 3 methods share identical data, seed, code and runtime.
    historical_com = None
    if method == "com" and seed == 42:
        historical_com = {
            "reference_official_correct": 18608,
            "reference_handmade_correct": 1974,
            "current_official_correct": official_metrics["all"]["correct"],
            "current_handmade_correct": handmade_metrics["all"]["correct"],
            "official_correct_delta": official_metrics["all"]["correct"] - 18608,
            "handmade_correct_delta": handmade_metrics["all"]["correct"] - 1974,
            "exact_reference_match":
                official_metrics["all"]["correct"] == 18608 and
                handmade_metrics["all"]["correct"] == 1974,
        }
        print("HISTORICAL CoM CONTROL COMPARISON: " +
              json.dumps(historical_com), flush=True)

    cm = confusion_matrix(hm_y, pred_hm, labels=np.arange(16))
    save_csv(out / "confusion_matrix.csv",
             [{"actual": LABELS[i], **{LABELS[j]: int(cm[i, j]) for j in range(16)}}
              for i in range(16)])
    save_csv(out / "pair_errors.csv", [
        {"actual": a, "predicted": b, "count": int(cm[LABELS.index(a), LABELS.index(b)])}
        for a, b in PAIRS
    ])
    precision, recall, f1, support = precision_recall_fscore_support(
        hm_y, pred_hm, labels=np.arange(16), zero_division=0)
    save_csv(out / "per_class.csv", [
        {"class": LABELS[k], "recall": float(recall[k]),
         "precision": float(precision[k]), "f1": float(f1[k]), "support": int(support[k])}
        for k in range(16)
    ])
    with gzip.open(out / "handmade_predictions.csv.gz", "wt", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["writer", "index", "true", "pred", "correct"])
        for k in range(2560):
            writer.writerow([int(writers[k]), k % 640, LABELS[int(hm_y[k])],
                             LABELS[int(pred_hm[k])], int(hm_y[k] == pred_hm[k])])

    save_csv(out / "loss_curve.csv",
             [{"iteration": i, "loss": float(loss)} for i, loss in
              enumerate(estimator.named_steps["model"].loss_curve_, 1)])
    model_path = out / "model.joblib"
    dump(estimator, model_path, compress=3)
    before_train = image_geometry(official["X_train"])
    after_train = image_geometry(train_x)
    before_hm = image_geometry(hm_raw)
    after_hm = image_geometry(hm_x)
    geometry = {
        "train_longside_std_before": float(np.std(before_train["long_side"])),
        "train_longside_std_after": float(np.std(after_train["long_side"])),
        "handmade_com_mean_offset_before": float(np.mean(before_hm["com_offset"])),
        "handmade_com_mean_offset_after": float(np.mean(after_hm["com_offset"])),
        "handmade_mass_before": float(np.sum(hm_raw)),
        "handmade_mass_after": float(np.sum(hm_x)),
    }
    result = {
        "experiment": "Normal-HypB-EXP006",
        "status": "trained",
        "method": method, "seed": seed,
        "target_longside_pixels": target,
        "foreground_threshold": 32,
        "size": "Combined EXP-CMB-004 apply_size: BBox-centered scipy affine_transform, order=1",
        "centering": "Combined EXP-CMB-004 center_batch: CoM to (13.5,13.5), integer shift",
        "pipeline": "process(raw) -> PixelScaler255 -> MLPClassifier(random_state=seed)",
        "model_hyperparameters": {"random_state": seed, "other": "sklearn MLPClassifier defaults"},
        "n_training": 76800, "n_official_validation": 19200, "n_handmade_dev": 2560,
        "official_test_used": False, "professor_test_used": False,
        "handmade_used_in_fit": False, "target_fitted_on_train_only": True,
        "official_sha256": official["sha256"],
        "writer_sha256": {str(w): hm[w]["sha256"] for w in WRITERS},
        "numpy": np.__version__, "scipy": scipy.__version__, "sklearn": sklearn.__version__,
        "python": platform.python_version(), "fit_seconds": float(fit_seconds),
        "mlp_iterations": int(estimator.named_steps["model"].n_iter_),
        "fit_warnings": [f"{v.category.__name__}: {v.message}" for v in caught],
        "official_validation": official_metrics,
        "handmade": handmade_metrics,
        "historical_com_reference_comparison": historical_com,
        "geometry": geometry,
        "model_sha256": sha256(model_path),
    }
    (out / "run_manifest.json").write_text(
        json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print("COMPLETE", json.dumps({"method": method, "seed": seed,
        "official": official_metrics["all"], "handmade": handmade_metrics},
        ensure_ascii=False), flush=True)


def aggregate(inputs, out):
    inputs = Path(inputs)
    out = Path(out)
    if out.exists():
        raise RuntimeError("Refusing to overwrite: " + str(out))
    out.mkdir(parents=True)
    files = sorted(inputs.glob("normal-exp006-*/run_manifest.json"))
    if len(files) != 9:
        raise RuntimeError(f"Expected 9 run manifests, got {len(files)}")
    observed = {}
    summaries = []
    for p in files:
        data = json.loads(p.read_text(encoding="utf-8"))
        method, seed = data["method"], int(data["seed"])
        if (method, seed) in observed:
            raise ValueError("Duplicate run: " + str((method,seed)))
        observed[method, seed] = p.parent
        if data["official_test_used"] or data["professor_test_used"] or data["handmade_used_in_fit"]:
            raise AssertionError("Data boundary violated")
        for group, metrics in data["handmade"].items():
            summaries.append({"method": method, "seed": seed, "dataset": "handmade",
                "group": group, **metrics})
        for group, metrics in data["official_validation"].items():
            summaries.append({"method": method, "seed": seed, "dataset": "official_val",
                "group": group, **metrics})
    if set(observed) != {(m,seed) for m in METHODS for seed in (42,43,44)}:
        raise AssertionError("Method-seed mismatch")
    save_csv(out / "all_metrics.csv", summaries)

    def read_predictions(path):
        with gzip.open(path, "rt", encoding="utf-8", newline="") as f:
            return {(int(r["writer"]), int(r["index"])): r
                    for r in csv.DictReader(f)}
    paired = []
    for seed in (42,43,44):
        base = read_predictions(observed["com",seed] / "handmade_predictions.csv.gz")
        for compared in ("size", "size_com"):
            other = read_predictions(observed[compared,seed] / "handmade_predictions.csv.gz")
            if set(base) != set(other) or len(base) != 2560:
                raise AssertionError("Paired prediction index mismatch")
            groups = {"all": [], "digits": [], "letters": []}
            for key in sorted(base):
                a, b = base[key], other[key]
                if a["true"] != b["true"]:
                    raise AssertionError("Label mismatch")
                grp = "digits" if a["true"] in "0123456789" else "letters"
                groups["all"].append((a,b))
                groups[grp].append((a,b))
            for grp, rows in groups.items():
                repaired = sum(int(a["correct"]) == 0 and int(b["correct"]) == 1 for a,b in rows)
                induced = sum(int(a["correct"]) == 1 and int(b["correct"]) == 0 for a,b in rows)
                paired.append({"seed":seed,"control":"com","candidate":compared,
                    "group":grp,"n":len(rows),"corrected":repaired,"new_errors":induced,
                    "net_correct":repaired-induced,
                    "delta_accuracy_pp":100*(repaired-induced)/len(rows)})
    save_csv(out / "paired_vs_com.csv", paired)

    lines = [
      "# EXP유006 — Normal Size/CoM comparison (review only)",
      "",
      "All values below are measurements from GitHub Actions; no Test data were used.",
      "Methods are CoM-only, BBox Size-only, and BBox Size -> CoM.",
      "Each model was freshly fitted on the SAME 76,800 official Train images,",
      "using sklearn MLPClassifier defaults and seeds 42,43,44.",
      "Target BBox long side 20px was fitted on official Training only.",
      "",
      "## Per-seed accuracy",
      "",
      "| Method | Seed | Official Val | Hand-made all | Digits | Letters |",
      "|---|---:|---:|---:|---:|---:|"
    ]
    for method in METHODS:
        for seed in (42,43,44):
            r=json.loads((observed[method,seed]/"run_manifest.json").read_text(encoding="utf-8"))
            o=r["official_validation"]["all"]["accuracy"]
            h=r["handmade"]
            lines.append(f"| {method} | {seed} | {100*o:.3f}% | "
                         f"{100*h['all']['accuracy']:.3f}% | "
                         f"{100*h['digits']['accuracy']:.3f}% | "
                         f"{100*h['letters']['accuracy']:.3f}% |")
    lines += [
      "", "## Paired Size+CoM vs CoM-only", "",
      "| Seed | Group | Corrections | Newly wrong | Net |",
      "|---|---|---:|---:|---:|"
    ]
    for r in paired:
        if r["candidate"] == "size_com":
            lines.append(f"| {r['seed']} | {r['group']} | {r['corrected']} | "
                         f"{r['new_errors']} | {r['net_correct']} |")
    lines += ["", "This is developer-data evidence, not an independent new-writer Test.",
              "Official Original Test and professor Test were never opened.",
              "Stroke normalization is OUT OF SCOPE. No model or pipeline is adopted.",
              "Review official Val, digits/letters and writer-level tradeoffs before next stage."]
    (out / "SUMMARY.md").write_text("\n".join(lines)+"\n", encoding="utf-8")
    print("EXP006 AGGREGATION COMPLETE", len(summaries), "summary rows", flush=True)


def main():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument("--method", choices=METHODS)
    p.add_argument("--seed", type=int, choices=(42,43,44))
    p.add_argument("--out", type=Path, required=True)
    p.add_argument("--aggregate", action="store_true")
    p.add_argument("--inputs", type=Path)
    a=p.parse_args()
    if a.aggregate:
        if not a.inputs:
            p.error("--aggregate requires --inputs")
        aggregate(a.inputs, a.out)
    else:
        if a.method is None or a.seed is None:
            p.error("--method and --seed are required")
        train(a.method, a.seed, a.out)

if __name__ == "__main__":
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")